# Lab 6a: Link Prediction & Sign Prediction (Heuristics First)
**Course:** IS353 - Graph and Network Analysis with Applications, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2025  

---

**Goals:**
- Implement **10 local** link prediction heuristics and **3 global** heuristics (Katz, Personalized PageRank similarity, Hitting time).
- Evaluate with **ROC-AUC, Average Precision, Precision@k**, and small plots.
- Implement **sign prediction** using local signed features + **logistic regression baseline**.
- (Optional) Run on a **larger SNAP dataset** (e.g., `facebook_combined.txt`).

**Environment**
- Python ≥ 3.9, `networkx`, `numpy`, `scipy`, `pandas`, `scikit-learn`, `matplotlib`.

> Tip: Keep the toy graph small (Karate Club) for quick iteration; then try the optional larger graph.


## 0. Setup

In [ ]:

import math
import random
from collections import defaultdict
from itertools import islice

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Matplotlib defaults (single-figure plots; no custom styles/colors per course policy)
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

rng = random.Random(42)
np.random.seed(42)


## 1. Evaluation Helpers

In [ ]:

def precision_at_k(y_true, y_score, k):
    order = np.argsort(y_score)[::-1]
    y_sorted = np.asarray(y_true)[order]
    k = min(k, len(y_sorted))
    return float(np.sum(y_sorted[:k])) / float(k) if k > 0 else 0.0

def eval_binary_scores(y_true, y_score, ks=(5,10,20)):
    auc = roc_auc_score(y_true, y_score)
    ap  = average_precision_score(y_true, y_score)
    out = {"AUC": auc, "AP": ap}
    for k in ks:
        out[f"P@{k}"] = precision_at_k(y_true, y_score, k)
    return out

def plot_roc_pr(y_true, y_score, title_prefix=""):
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, label="ROC")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"{title_prefix} ROC Curve")
    plt.legend()
    plt.show()

    # PR
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    plt.figure()
    plt.plot(recall, precision, label="PR")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} Precision-Recall Curve")
    plt.legend()
    plt.show()


## 2. Toy Graph & Holdout (Unsigned)

In [ ]:

# Use Zachary's Karate Club graph
G_full = nx.karate_club_graph()
G = G_full.copy()

# Hold out ~15% of edges as positive test examples
E = list(G.edges())
rng.shuffle(E)
n_hold = max(1, int(0.15 * len(E)))
E_test_pos = E[:n_hold]
G.remove_edges_from(E_test_pos)

# Sample same number of non-edges as negative test examples
non_edges = list(nx.non_edges(G))
rng.shuffle(non_edges)
E_test_neg = non_edges[:n_hold]

E_test = [(u, v, 1) for (u, v) in E_test_pos] + [(u, v, 0) for (u, v) in E_test_neg]
print(f"Train edges: {G.number_of_edges()}, Test positives: {len(E_test_pos)}, Test negatives: {len(E_test_neg)}")

pos = nx.spring_layout(G_full, seed=42)
plt.figure()
nx.draw(G_full, pos, with_labels=True, node_size=400)
plt.title("Zachary Karate Club (Full)")
plt.show()

plt.figure()
nx.draw(G, pos, with_labels=True, node_size=400)
plt.title("Train Graph (after holding out test positives)")
plt.show()


## 3. Ten Local Link Prediction Heuristics

In [ ]:

def deg(G, x):
    return G.degree(x)

def common_neighbors_count(G, x, y):
    # NetworkX common_neighbors works for simple graphs
    return len(list(nx.common_neighbors(G, x, y)))

def CN(G, x, y):
    return common_neighbors_count(G, x, y)

def JC(G, x, y):
    nxu = set(G.neighbors(x)); nyu = set(G.neighbors(y))
    inter = len(nxu & nyu); uni = len(nxu | nyu)
    return inter / uni if uni else 0.0

def AA(G, x, y):
    s = 0.0
    for z in nx.common_neighbors(G, x, y):
        dz = deg(G, z)
        if dz > 1:
            s += 1.0 / math.log(dz)
    return s

def PA(G, x, y):
    return deg(G, x) * deg(G, y)

def Salton(G, x, y):
    cn = CN(G, x, y)
    denom = math.sqrt(deg(G, x) * deg(G, y))
    return cn / denom if denom else 0.0

def Sorensen(G, x, y):
    cn = CN(G, x, y)
    denom = deg(G, x) + deg(G, y)
    return (2.0 * cn) / denom if denom else 0.0

def HPI(G, x, y):
    cn = CN(G, x, y)
    denom = min(deg(G, x), deg(G, y))
    return cn / denom if denom else 0.0

def HDI(G, x, y):
    cn = CN(G, x, y)
    denom = max(deg(G, x), deg(G, y))
    return cn / denom if denom else 0.0

def LHN1(G, x, y):
    cn = CN(G, x, y)
    denom = deg(G, x) * deg(G, y)
    return cn / denom if denom else 0.0

def RA(G, x, y):
    s = 0.0
    for z in nx.common_neighbors(G, x, y):
        dz = deg(G, z)
        if dz > 0:
            s += 1.0 / dz
    return s

LOCAL_KEYS = ["CN","JC","AA","PA","Salton","Sorensen","HPI","HDI","LHN1","RA"]

def compute_local_scores(G, pairs):
    rows = []
    for (u, v, y) in pairs:
        rows.append({
            "u": u, "v": v, "y": y,
            "CN": CN(G,u,v),
            "JC": JC(G,u,v),
            "AA": AA(G,u,v),
            "PA": PA(G,u,v),
            "Salton": Salton(G,u,v),
            "Sorensen": Sorensen(G,u,v),
            "HPI": HPI(G,u,v),
            "HDI": HDI(G,u,v),
            "LHN1": LHN1(G,u,v),
            "RA": RA(G,u,v),
        })
    return pd.DataFrame(rows)

df_local = compute_local_scores(G, E_test)
print(df_local.shape)
df_local.head()


**Katz Index**  
Counts weighted **walks** of all lengths between nodes with exponential damping $\beta^l$:
$(I - \beta A)^{-1} - I$, where $\beta < 1/\rho(A)$.

**Personalized PageRank (PPR) Similarity**  
$PPR_{u \to v}$ is the stationary probability of a random walk with teleportation to $u$. It measures the likelihood of being at node $v$ starting from $u$ and can be used as the similarity between $u$ and $v$. Since PPR is asymmetric, it may be useful to symmetrize it for undirected graphs by $PPR_{u \to v} + PPR_{v \to u}$.

**Hitting Time (HT)**  
For undirected connected graphs: $H_{i \to j} = \text{vol}(G) \cdot (L^+_{jj} - L^+_{ij})$, where $L^+$ is the Moore-Penrose pseudoinverse of the **combinatorial** Laplacian $L = D - A$. We convert HT to a **similarity** (lower time = higher similarity) by taking negation $-H_{i \to j}$ or inverse $1/H_{i \to j}$. Since HT is asymmetric, it may be useful to symmetrize it by using Commute Time $H_{i \to j} + H_{j \to i}$.

In [ ]:

import numpy.linalg as LA

# Helper: consistent node ordering
NODES = sorted(G.nodes())
IDX = {n:i for i,n in enumerate(NODES)}

A = nx.to_numpy_array(G, nodelist=NODES)  # adjacency
D = np.diag([G.degree(n) for n in NODES])
L = D - A  # combinatorial Laplacian

def katz_matrix(G, beta=0.03):
    # (I - beta A)^{-1} - I
    A = nx.to_numpy_array(G, nodelist=NODES)
    I = np.eye(A.shape[0])
    K = LA.inv(I - beta*A) - I
    return K

K = katz_matrix(G, beta=0.03)

def katz_score(u, v):
    i, j = IDX[u], IDX[v]
    # Symmetrize for undirected
    return K[i, j] + K[j, i]

def ppr_score(G, u, v, alpha=0.85):
    pers = {n: 0.0 for n in G}
    pers[u] = 1.0
    pr = nx.pagerank(G, alpha=alpha, personalization=pers)
    return pr.get(v, 0.0)

# Hitting time via Laplacian pseudoinverse
Lplus = LA.pinv(L)
volG = 2.0 * G.number_of_edges()  # sum of degrees

def hitting_time(u, v):
    i, j = IDX[u], IDX[v]
    return volG * (Lplus[j, j] - Lplus[i, j])  # H_{u->v}

# Convert HT to similarity (lower is better): use negative z-score
def hitting_time_similarity(u, v, mean_ht=None, std_ht=None):
    ht = hitting_time(u, v)
    if mean_ht is not None and std_ht is not None and std_ht > 0:
        z = (ht - mean_ht) / std_ht
        return -z
    return -ht  # fallback


In [ ]:

# Compute global scores for all test pairs
# Precompute mean/std for HT to make scale comparable
ht_values = [hitting_time(u, v) for (u, v, _) in E_test]
mean_ht, std_ht = float(np.mean(ht_values)), float(np.std(ht_values))

rows = []
for _, row in df_local.iterrows():
    u, v, y = row["u"], row["v"], row["y"]
    katz = katz_score(u, v)
    ppr_uv = ppr_score(G, u, v)
    ppr_vu = ppr_score(G, v, u)
    ppr_sym = ppr_uv + ppr_vu
    ht_sim = hitting_time_similarity(u, v, mean_ht, std_ht)
    rec = dict(row)
    rec.update({"Katz": katz, "PPR(u->v)": ppr_uv, "PPR(v->u)": ppr_vu, "PPR_sym": ppr_sym, "HT_sim": ht_sim})
    rows.append(rec)

df_all = pd.DataFrame(rows)
df_all.head()


## 5. Evaluation (ROC-AUC, AP, Precision@k)

In [ ]:

keys = ["CN","JC","AA","PA","Salton","Sorensen","HPI","HDI","LHN1","RA","Katz","PPR_sym","HT_sim"]
y_true = df_all["y"].values

summary = []
for k in keys:
    metrics = eval_binary_scores(y_true, df_all[k].values, ks=(5,10,20))
    row = {"Heuristic": k}
    row.update(metrics)
    summary.append(row)

df_eval = pd.DataFrame(summary).sort_values("AUC", ascending=False)
df_eval


In [ ]:

# Plot ROC/PR for top-3 by AUC
top3 = df_eval.head(3)["Heuristic"].tolist()
for k in top3:
    plot_roc_pr(y_true, df_all[k].values, title_prefix=k)


## 6. Sign Prediction (Toy Signed Graph)

We construct a small **signed** undirected graph and split edges into train/test.
Features: **signed common-neighbor patterns** around $(u,v)$: counts of ++, +−, −+, −− among common neighbors (and total CN).
We train a **logistic regression** to predict edge sign (+1 vs −1).

In [ ]:

# Build a toy signed graph (undirected)
SG = nx.Graph()
SG.add_nodes_from(range(1, 13))

# Create two friendly clusters with a few antagonistic cross-links
pos_edges = [(1,2),(2,3),(3,4),(4,1),(2,4),
             (5,6),(6,7),(7,8),(8,5),(6,8),
             (9,10),(10,11),(11,12),(12,9),(10,12)]
neg_edges = [(3,6),(4,7),(1,8),(2,5), (9,5),(11,3)]  # cross-group negatives

for u,v in pos_edges:
    SG.add_edge(u,v, sign=+1)
for u,v in neg_edges:
    SG.add_edge(u,v, sign=-1)

print("Signed graph:", SG.number_of_nodes(), "nodes,", SG.number_of_edges(), "edges")
pos_s = nx.spring_layout(SG, seed=7)
plt.figure()
edge_colors = ["green" if SG[u][v]["sign"]>0 else "red" for u,v in SG.edges()]
nx.draw(SG, pos_s, with_labels=True, node_size=400, edge_color=edge_colors)
plt.title("Toy Signed Graph (+ green, − red)")
plt.show()


In [ ]:

# Split edges into train/test
all_edges = list(SG.edges(data=True))
rng.shuffle(all_edges)
n_test = max(2, int(0.3 * len(all_edges)))
test_edges = all_edges[:n_test]
train_edges = all_edges[n_test:]

def signed_cn_features(SG, u, v):
    cn = set(nx.common_neighbors(SG, u, v))
    pp = pm = mp = mm = 0
    for z in cn:
        s_uz = SG[u][z]["sign"] if SG.has_edge(u,z) else 0
        s_vz = SG[v][z]["sign"] if SG.has_edge(v,z) else 0
        if s_uz == +1 and s_vz == +1: pp += 1
        elif s_uz == +1 and s_vz == -1: pm += 1
        elif s_uz == -1 and s_vz == +1: mp += 1
        elif s_uz == -1 and s_vz == -1: mm += 1
    return np.array([pp, pm, mp, mm, len(cn)], dtype=float)

def edge_list_to_xy(SG, edges):
    X, y = [], []
    for u, v, data in edges:
        X.append(signed_cn_features(SG, u, v))
        y.append(1 if data["sign"] > 0 else 0)
    return np.vstack(X), np.array(y)

Xtr, ytr = edge_list_to_xy(SG, train_edges)
Xte, yte = edge_list_to_xy(SG, test_edges)

clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
yp = clf.predict(Xte)
proba = clf.predict_proba(Xte)[:,1]
print("Sign Prediction — Accuracy:", accuracy_score(yte, yp), "F1:", f1_score(yte, yp))

# Show learned coefficients
coef_names = ["pp","pm","mp","mm","CN"]
print("Feature weights:", dict(zip(coef_names, clf.coef_[0])))

# Small PR/ROC plots for sign model
plot_roc_pr(yte, proba, title_prefix="Sign (LogReg)")


## 7. (Optional) Larger Graph — SNAP `facebook_combined.txt`


**Instructions**
1. Download `facebook_combined.txt` from SNAP and place it next to this notebook.
2. Each line: `u v` (undirected edge). Node IDs are integers.
3. We will hide 1-2% edges for test, sample equal non-edges, and recompute a subset of heuristics.

> For very large graphs: skip exact **Katz** (matrix inversion is $O(n^3)$). Prefer **PPR** only, or approximate Katz with truncated powers.


In [ ]:

# Uncomment and run if you have the file locally.
# import os
# path = "facebook_combined.txt"
# assert os.path.exists(path), "Place facebook_combined.txt in the working directory."

# Gbig = nx.read_edgelist(path, nodetype=int)
# print("Loaded:", Gbig.number_of_nodes(), "nodes,", Gbig.number_of_edges(), "edges")

# # Build a small evaluation split
# E = list(Gbig.edges())
# rng.shuffle(E)
# n_hold = max(50, int(0.01 * len(E)))
# E_test_pos = E[:n_hold]
# Gtrain = Gbig.copy()
# Gtrain.remove_edges_from(E_test_pos)

# non_edges = list(nx.non_edges(Gtrain))
# rng.shuffle(non_edges)
# E_test_neg = non_edges[:n_hold]
# E_test_big = [(u,v,1) for u,v in E_test_pos] + [(u,v,0) for u,v in E_test_neg]

# # Compute a subset: 10 locals + PPR_sym (skip Katz & HT for speed)
# df_local_big = compute_local_scores(Gtrain, E_test_big)

# def ppr_sym_big(G, u, v, alpha=0.85):
#     pers_u = {n:0.0 for n in G}; pers_u[u]=1.0
#     pr_u = nx.pagerank(G, alpha=alpha, personalization=pers_u)
#     pers_v = {n:0.0 for n in G}; pers_v[v]=1.0
#     pr_v = nx.pagerank(G, alpha=alpha, personalization=pers_v)
#     return pr_u.get(v,0.0) + pr_v.get(u,0.0)

# rows = []
# for _, r in df_local_big.iterrows():
#     u, v, y = r["u"], r["v"], r["y"]
#     r2 = dict(r)
#     r2["PPR_sym"] = ppr_sym_big(Gtrain, u, v)
#     rows.append(r2)
# df_big = pd.DataFrame(rows)

# keys_big = ["CN","JC","AA","PA","Salton","Sorensen","HPI","HDI","LHN1","RA","PPR_sym"]
# y_true_big = df_big["y"].values
# summary_big = []
# for k in keys_big:
#     metrics = eval_binary_scores(y_true_big, df_big[k].values, ks=(10,50,100))
#     row = {"Heuristic": k}; row.update(metrics); summary_big.append(row)
# pd.DataFrame(summary_big).sort_values("AUC", ascending=False)


## 8. Questions


- Which heuristics performed best on the toy graph? Why?
- Did **degree-normalized** indices (Salton/Sørensen/LHN1, etc.) beat raw **CN**?
- Is **RA** better than **AA** here? Why?
- Are **global** methods (Katz/PPR/HT) worth their cost?
- For **signs**, which features mattered most (pp, pm, mp, mm, CN)?
- (Optional) Run and analyze on a larger graph (Note: the conclusion may be very different with larger graphs/other domains).